# Distributed Multi-Rank Stencil & Asynchronous Halo Exchange in MPI + CUDA (Python)

**Ayazgaliyev Aibek · Student ID 230103341 · GitHub `Stierlitzx` · 19 November 2026**

4-rank (2 × 2 non-periodic Cartesian) anisotropic reaction-diffusion solver on a 512 × 512 grid.
Each rank owns a 258 × 258 padded device array, exchanges halos through GPU pack/unpack kernels
and pinned host buffers with `Isend`/`Irecv`, and updates cells with an 18 × 18 shared-memory
tiled stencil on two CUDA streams (deep interior vs. boundary ring).

Run order: **Runtime → Run all** on a T4 GPU runtime (Colab). The launcher cell also works with
MS-MPI `mpiexec` on Windows for local validation.

In [ ]:
# CELL 1 - ENVIRONMENT VERIFICATION & MPI4PY INSTALLATION
# (portable form of the three ! commands: works on Colab/Linux and on Windows + MS-MPI)
import shutil, subprocess, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,uuid,compute_cap,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
if shutil.which("mpirun"):
    print("\n".join(subprocess.run(["mpirun", "--version"], capture_output=True, text=True)
                    .stdout.splitlines()[:2]))
elif shutil.which("mpiexec"):
    print([l for l in subprocess.run(["mpiexec", "-help"], capture_output=True, text=True)
           .stdout.splitlines() if "Version" in l][0])
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mpi4py", "numba", "numpy",
                    "numba-cuda[cu12]"], check=True)
import mpi4py, numba, numpy
print(f"python {sys.version.split()[0]} | numpy {numpy.__version__} | numba {numba.__version__} "
      f"| mpi4py {mpi4py.__version__}")

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-198f4157-3506-5952-ab80-80b6dd077a59, 7.5, 15360 MiB

mpirun (Open MPI) 4.1.6

python 3.13.16 | numpy 2.4.6 | numba 0.64.0 | mpi4py 4.1.2


In [ ]:
# CELL 2 - STUDENT IDENTITY CONFIGURATION
# Replace with your exact GitHub username before running the solver
GITHUB_USERNAME = "Stierlitzx"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "You must set GITHUB_USERNAME to your real GitHub handle."
)
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: Stierlitzx


In [ ]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import hashlib
import subprocess
import time
import re
import warnings
import numpy as np

# Small border-ring launches (1x16 / 16x1 blocks) are intentional; silence Numba's occupancy hint.
os.environ.setdefault("NUMBA_CUDA_LOW_OCCUPANCY_WARNINGS", "0")
warnings.filterwarnings("ignore", category=DeprecationWarning)

from mpi4py import MPI
from numba import cuda, float64

# Compile-time constants for 2D shared-memory tiling (16x16 interior + 1-cell halo apron)
TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15

# Additional solver constants (not part of the instructor schema, recorded in the manifest)
DIAG_EVERY = 100
CFL_SAFETY = 0.90          # dt = CFL_SAFETY * dx^2 / (4 * kappa_max)
WARMUP_STEPS = 3           # JIT-compile + warm caches before timed runs
PACK_THREADS = 128

# Message tags = direction of travel of the payload
TAG_TO_UP, TAG_TO_DOWN, TAG_TO_LEFT, TAG_TO_RIGHT = 10, 11, 12, 13
# Row index of each strip inside the (4, n) send / receive buffers
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3


def get_gpu_uuid() -> str:
    """Query active NVIDIA GPU UUID via nvidia-smi."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def get_gpu_name() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Derive deterministic 64-bit integer seed bound to student GitHub handle and GPU UUID."""
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16)


def log0(ctx, *args):
    if ctx["rank"] == 0:
        print(*args, flush=True)


# ==============================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context Initialization
# ==============================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {size}.")

    # Bind process to GPU 0 on Google Colab
    cuda.select_device(0)

    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    gpu_name = get_gpu_name() if rank == 0 else None
    gpu_name = comm.bcast(gpu_name, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    dims = [2, 2]
    periods = [False, False]
    cart_comm = comm.Create_cart(dims=dims, periods=periods, reorder=False)
    coords = cart_comm.Get_coords(cart_comm.Get_rank())
    nbr_up, nbr_down = cart_comm.Shift(0, 1)      # axis 0: rows (y direction)
    nbr_left, nbr_right = cart_comm.Shift(1, 1)   # axis 1: columns (x direction)

    ny = GLOBAL_NY // dims[0]
    nx = GLOBAL_NX // dims[1]
    return {
        "comm": comm,
        "cart_comm": cart_comm,
        "rank": cart_comm.Get_rank(),
        "size": size,
        "dims": dims,
        "periods": periods,
        "coords": tuple(int(c) for c in coords),
        "nbr_up": nbr_up,
        "nbr_down": nbr_down,
        "nbr_left": nbr_left,
        "nbr_right": nbr_right,
        "ny": ny,
        "nx": nx,
        "github_user": github_user,
        "gpu_uuid": gpu_uuid,
        "gpu_name": gpu_name,
        "instance_seed": instance_seed,
    }


def rank_table(ctx):
    """Collective per-rank metadata table built with Allreduce (no field arrays are collected)."""
    cart = ctx["cart_comm"]
    pn = MPI.PROC_NULL
    row = np.zeros((ctx["size"], 9), dtype=np.float64)
    nbrs = [ctx["nbr_up"], ctx["nbr_down"], ctx["nbr_left"], ctx["nbr_right"]]
    row[ctx["rank"]] = [
        ctx["coords"][0], ctx["coords"][1],
        *[(-1 if n == pn else n) for n in nbrs],
        ctx["ny"] + 2, ctx["nx"] + 2, os.getpid(),
    ]
    table = np.zeros_like(row)
    cart.Allreduce(row, table, op=MPI.SUM)
    return table


# ==============================================================================
# TASK 2: Seeded Heterogeneous Subdomain Initialization
# ==============================================================================
def seeded_parameters(instance_seed: int):
    """All ranks derive identical physical parameters from the same 64-bit seed."""
    rng = np.random.Generator(np.random.PCG64(instance_seed))
    n_pulses = 3
    params = {
        "pulse_cx": rng.uniform(0.22, 0.78, n_pulses).tolist(),
        "pulse_cy": rng.uniform(0.22, 0.78, n_pulses).tolist(),
        "pulse_amp": rng.uniform(0.60, 1.40, n_pulses).tolist(),
        "pulse_sigma": rng.uniform(0.040, 0.070, n_pulses).tolist(),
        "kappa_fx": int(rng.integers(1, 4)),
        "kappa_fy": int(rng.integers(1, 4)),
        "kappa_phx": float(rng.uniform(0.0, 1.0)),
        "kappa_phy": float(rng.uniform(0.0, 1.0)),
        # Localised source/sink dipole (equal and opposite lobes -> zero net injection)
        "src_amp": float(rng.uniform(3.0, 6.0)),
        "src_sigma": 0.030,
        "src_pos": [float(v) for v in rng.uniform(0.30, 0.70, 2)],
        "sink_pos": [float(v) for v in rng.uniform(0.30, 0.70, 2)],
    }
    return params


def initialize_subdomains(ctx, params):
    """Build rank-local (ny+2, nx+2) host fields from analytic global coordinates.

    No global 512x512 array is ever allocated. Local index li in 0..ny+1 maps to global
    index gi = row_rank * ny + li, with physical coordinate y = gi * dy (dy = 1/513), so
    gi = 0 and gi = 513 are the Dirichlet walls and 1..512 are the interior cells.
    """
    ny, nx = ctx["ny"], ctx["nx"]
    rr, cr = ctx["coords"]
    dx = 1.0 / (GLOBAL_NX + 1)
    dy = 1.0 / (GLOBAL_NY + 1)
    gi = rr * ny + np.arange(ny + 2, dtype=np.float64)
    gj = cr * nx + np.arange(nx + 2, dtype=np.float64)
    Y, X = np.meshgrid(gi * dy, gj * dx, indexing="ij")

    p = params
    kappa = 0.16 + 0.08 * np.sin(2.0 * np.pi * (p["kappa_fx"] * X + p["kappa_phx"])) \
                       * np.cos(2.0 * np.pi * (p["kappa_fy"] * Y + p["kappa_phy"]))

    u = np.zeros((ny + 2, nx + 2), dtype=np.float64)
    for cx, cy, a, s in zip(p["pulse_cx"], p["pulse_cy"], p["pulse_amp"], p["pulse_sigma"]):
        u += a * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2.0 * s * s))

    s2 = 2.0 * p["src_sigma"] ** 2
    src = p["src_amp"] * (
        np.exp(-((X - p["src_pos"][0]) ** 2 + (Y - p["src_pos"][1]) ** 2) / s2)
        - np.exp(-((X - p["sink_pos"][0]) ** 2 + (Y - p["sink_pos"][1]) ** 2) / s2)
    )

    # Dirichlet u = 0 on outer global edges (neighbour == MPI.PROC_NULL)
    if ctx["nbr_up"] == MPI.PROC_NULL:
        u[0, :] = 0.0
    if ctx["nbr_down"] == MPI.PROC_NULL:
        u[ny + 1, :] = 0.0
    if ctx["nbr_left"] == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if ctx["nbr_right"] == MPI.PROC_NULL:
        u[:, nx + 1] = 0.0

    return {
        "u": np.ascontiguousarray(u),
        "kappa": np.ascontiguousarray(kappa),
        "src": np.ascontiguousarray(src),
        "dx": dx,
        "dy": dy,
    }


# ==============================================================================
# TASK 3: GPU Halo Pack/Unpack Kernels & Pinned MPI Exchange
# ==============================================================================
@cuda.jit
def pack_halos_kernel(u, send_buf, ny, nx):
    """Gather the four owned border strips (two of them strided columns) into contiguous rows."""
    k = cuda.grid(1)
    if k < nx:
        send_buf[0, k] = u[1, k + 1]       # top owned row     -> neighbour up
        send_buf[1, k] = u[ny, k + 1]      # bottom owned row  -> neighbour down
    if k < ny:
        send_buf[2, k] = u[k + 1, 1]       # left owned column  (stride nx+2) -> neighbour left
        send_buf[3, k] = u[k + 1, nx]      # right owned column (stride nx+2) -> neighbour right


@cuda.jit
def unpack_halos_kernel(u, recv_buf, ny, nx, has_up, has_down, has_left, has_right):
    """Scatter received contiguous vectors into ghost rows/columns 0 and n+1."""
    k = cuda.grid(1)
    if k < nx:
        if has_up:
            u[0, k + 1] = recv_buf[0, k]
        if has_down:
            u[ny + 1, k + 1] = recv_buf[1, k]
    if k < ny:
        if has_left:
            u[k + 1, 0] = recv_buf[2, k]
        if has_right:
            u[k + 1, nx + 1] = recv_buf[3, k]


def allocate_exchange_buffers(ctx):
    n = max(ctx["ny"], ctx["nx"])
    bufs = {
        "d_send": cuda.device_array((4, n), dtype=np.float64),
        "d_recv": cuda.to_device(np.zeros((4, n), dtype=np.float64)),
        "h_send": cuda.pinned_array((4, n), dtype=np.float64),
        "h_recv": cuda.pinned_array((4, n), dtype=np.float64),
    }
    bufs["h_send"][:] = 0.0
    bufs["h_recv"][:] = 0.0
    return bufs


def exchange_halos_async(ctx, bufs, d_u, stream):
    """Stage A + Stage B (posting): GPU pack -> async D2H into pinned memory -> Irecv/Isend.

    Returns the list of outstanding MPI requests; the caller overlaps work and then calls
    complete_halo_exchange().
    """
    ny, nx = ctx["ny"], ctx["nx"]
    cart = ctx["cart_comm"]
    n = bufs["d_send"].shape[1]
    blocks = (n + PACK_THREADS - 1) // PACK_THREADS
    pack_halos_kernel[blocks, PACK_THREADS, stream](d_u, bufs["d_send"], ny, nx)
    bufs["d_send"].copy_to_host(bufs["h_send"], stream=stream)
    stream.synchronize()  # pinned send buffers are now valid on the host

    hs, hr = bufs["h_send"], bufs["h_recv"]
    reqs = [
        # receive what neighbours sent towards me
        cart.Irecv([hr[UP, :nx], MPI.DOUBLE], source=ctx["nbr_up"], tag=TAG_TO_DOWN),
        cart.Irecv([hr[DOWN, :nx], MPI.DOUBLE], source=ctx["nbr_down"], tag=TAG_TO_UP),
        cart.Irecv([hr[LEFT, :ny], MPI.DOUBLE], source=ctx["nbr_left"], tag=TAG_TO_RIGHT),
        cart.Irecv([hr[RIGHT, :ny], MPI.DOUBLE], source=ctx["nbr_right"], tag=TAG_TO_LEFT),
        # send my owned border strips
        cart.Isend([hs[UP, :nx], MPI.DOUBLE], dest=ctx["nbr_up"], tag=TAG_TO_UP),
        cart.Isend([hs[DOWN, :nx], MPI.DOUBLE], dest=ctx["nbr_down"], tag=TAG_TO_DOWN),
        cart.Isend([hs[LEFT, :ny], MPI.DOUBLE], dest=ctx["nbr_left"], tag=TAG_TO_LEFT),
        cart.Isend([hs[RIGHT, :ny], MPI.DOUBLE], dest=ctx["nbr_right"], tag=TAG_TO_RIGHT),
    ]
    return reqs


def complete_halo_exchange(ctx, bufs, d_u, reqs, stream):
    """Stage C (part 1): Waitall -> async H2D of pinned halos -> GPU unpack into ghosts."""
    MPI.Request.Waitall(reqs)
    bufs["d_recv"].copy_to_device(bufs["h_recv"], stream=stream)
    n = bufs["d_recv"].shape[1]
    blocks = (n + PACK_THREADS - 1) // PACK_THREADS
    pn = MPI.PROC_NULL
    unpack_halos_kernel[blocks, PACK_THREADS, stream](
        d_u, bufs["d_recv"], ctx["ny"], ctx["nx"],
        int(ctx["nbr_up"] != pn), int(ctx["nbr_down"] != pn),
        int(ctx["nbr_left"] != pn), int(ctx["nbr_right"] != pn),
    )


# ==============================================================================
# TASK 4: 2D Shared-Memory Tiled Stencil & Dual-Stream Orchestrator
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, u_new, kappa, src, dt, inv_dx2, inv_dy2, lam,
                         r_start, r_end, c_start, c_end):
    """Update u_new[r_start..r_end, c_start..c_end] (inclusive) with an 18x18 shared tile.

    Reads are confined to the cross-shaped neighbourhood of the box, so a launch on the
    deep interior [2..n-1]^2 never touches ghost cells 0 / n+1.
    """
    smem = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty   # global row in padded array
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx   # global column in padded array
    sy = ty + 1
    sx = tx + 1

    # Centre cell (also serves as apron for a neighbour when the tile is partial)
    if i <= r_end + 1 and j <= c_end + 1:
        smem[sy, sx] = u[i, j]
    # 1-cell halo apron loaded by the tile's edge threads
    if ty == 0 and j <= c_end:
        smem[0, sx] = u[i - 1, j]
    if ty == TILE_DIM - 1 and i + 1 <= r_end + 1 and j <= c_end:
        smem[SHARED_DIM - 1, sx] = u[i + 1, j]
    if tx == 0 and i <= r_end:
        smem[sy, 0] = u[i, j - 1]
    if tx == TILE_DIM - 1 and j + 1 <= c_end + 1 and i <= r_end:
        smem[sy, SHARED_DIM - 1] = u[i, j + 1]

    cuda.syncthreads()

    if i <= r_end and j <= c_end:
        c = smem[sy, sx]
        kc = kappa[i, j]
        k_n = 0.5 * (kappa[i - 1, j] + kc)
        k_s = 0.5 * (kappa[i + 1, j] + kc)
        k_w = 0.5 * (kappa[i, j - 1] + kc)
        k_e = 0.5 * (kappa[i, j + 1] + kc)
        div_y = (k_s * (smem[sy + 1, sx] - c) - k_n * (c - smem[sy - 1, sx])) * inv_dy2
        div_x = (k_e * (smem[sy, sx + 1] - c) - k_w * (c - smem[sy, sx - 1])) * inv_dx2
        u_new[i, j] = c + dt * (div_x + div_y - lam * c * c * c + src[i, j])


def launch_stencil(state, stream, r0, r1, c0, c1):
    rows = r1 - r0 + 1
    cols = c1 - c0 + 1
    grid = ((cols + TILE_DIM - 1) // TILE_DIM, (rows + TILE_DIM - 1) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        state["d_u"], state["d_u_new"], state["d_kappa"], state["d_src"],
        state["dt"], state["inv_dx2"], state["inv_dy2"], LAMBDA_DAMP,
        r0, r1, c0, c1,
    )


def step_serialized(ctx, state, bufs, streams, timers):
    """Reference path: exchange fully, then one full-domain stencil on a single stream."""
    s = streams["boundary"]
    ny, nx = ctx["ny"], ctx["nx"]
    e0, e1 = timers["ev"][0], timers["ev"][1]
    e0.record(s)
    reqs = exchange_halos_async(ctx, bufs, state["d_u"], s)
    complete_halo_exchange(ctx, bufs, state["d_u"], reqs, s)
    launch_stencil(state, s, 1, ny, 1, nx)
    e1.record(s)
    s.synchronize()
    timers["full_ms"] += cuda.event_elapsed_time(e0, e1)
    state["d_u"], state["d_u_new"] = state["d_u_new"], state["d_u"]


def step_overlapped(ctx, state, bufs, streams, timers):
    """Dual-stream pipeline: Stage A/B on stream_boundary, deep interior on stream_interior."""
    sb, si = streams["boundary"], streams["interior"]
    ny, nx = ctx["ny"], ctx["nx"]
    ev = timers["ev"]

    ev[0].record(sb)
    reqs = exchange_halos_async(ctx, bufs, state["d_u"], sb)           # Stage A (+ post B)
    ev[1].record(sb)

    ev[2].record(si)
    launch_stencil(state, si, 2, ny - 1, 2, nx - 1)                     # Stage B: deep interior
    ev[3].record(si)

    ev[4].record(sb)
    complete_halo_exchange(ctx, bufs, state["d_u"], reqs, sb)          # Stage C: wait/H2D/unpack
    launch_stencil(state, sb, 1, 1, 1, nx)                              # top ring row
    launch_stencil(state, sb, ny, ny, 1, nx)                            # bottom ring row
    launch_stencil(state, sb, 2, ny - 1, 1, 1)                          # left ring column
    launch_stencil(state, sb, 2, ny - 1, nx, nx)                        # right ring column
    ev[5].record(sb)

    sb.synchronize()
    si.synchronize()
    timers["pack_ms"] += cuda.event_elapsed_time(ev[0], ev[1])
    timers["interior_ms"] += cuda.event_elapsed_time(ev[2], ev[3])
    timers["boundary_ms"] += cuda.event_elapsed_time(ev[4], ev[5])
    state["d_u"], state["d_u_new"] = state["d_u_new"], state["d_u"]


# ==============================================================================
# TASK 5: Distributed Reductions, PTX Audit & Comparative Benchmark
# ==============================================================================
def make_state(ctx, fields, dt):
    return {
        "d_u": cuda.to_device(fields["u"]),
        "d_u_new": cuda.to_device(fields["u"]),   # carries identical Dirichlet ghosts
        "d_kappa": cuda.to_device(fields["kappa"]),
        "d_src": cuda.to_device(fields["src"]),
        "dt": float(dt),
        "inv_dx2": 1.0 / fields["dx"] ** 2,
        "inv_dy2": 1.0 / fields["dy"] ** 2,
    }


def global_diagnostics(ctx, state, fields, step):
    """Energy E = sum(u dx dy), L2 residual ||u^n - u^(n-1)||_2, max|u| via Allreduce."""
    ny, nx = ctx["ny"], ctx["nx"]
    u = state["d_u"].copy_to_host()[1:ny + 1, 1:nx + 1]
    u_prev = state["d_u_new"].copy_to_host()[1:ny + 1, 1:nx + 1]
    local = np.array([
        u.sum() * fields["dx"] * fields["dy"],
        0.0 if step == 0 else float(((u - u_prev) ** 2).sum()),
    ])
    glob = np.zeros(2)
    ctx["cart_comm"].Allreduce(local, glob, op=MPI.SUM)
    umax_l = np.array([np.abs(u).max()])
    umax_g = np.zeros(1)
    ctx["cart_comm"].Allreduce(umax_l, umax_g, op=MPI.MAX)
    return {"step": step, "energy": float(glob[0]), "l2_residual": float(math.sqrt(glob[1])),
            "u_max": float(umax_g[0])}


def run_mode(ctx, fields, dt, bufs, streams, mode, steps, record=True):
    state = make_state(ctx, fields, dt)
    timers = {"ev": [cuda.event(timing=True) for _ in range(6)],
              "full_ms": 0.0, "pack_ms": 0.0, "interior_ms": 0.0, "boundary_ms": 0.0}
    step_fn = step_serialized if mode == "serialized" else step_overlapped
    trace = []
    if record:
        trace.append(global_diagnostics(ctx, state, fields, 0))
    ctx["cart_comm"].Barrier()
    t_start = MPI.Wtime()
    for n in range(1, steps + 1):
        step_fn(ctx, state, bufs, streams, timers)
        if record and n % DIAG_EVERY == 0:
            trace.append(global_diagnostics(ctx, state, fields, n))
    ctx["cart_comm"].Barrier()
    wall_ms = (MPI.Wtime() - t_start) * 1e3
    timers.pop("ev")
    return state, trace, wall_ms, timers


def ptx_audit():
    asm = tiled_stencil_kernel.inspect_asm()
    ptx = "\n".join(asm.values()) if isinstance(asm, dict) else str(asm)
    counts = {
        "st.shared.f64": len(re.findall(r"\bst\.shared\.f64\b", ptx)),
        "ld.shared.f64": len(re.findall(r"\bld\.shared\.f64\b", ptx)),
        "bar.sync": len(re.findall(r"\bbar\.sync\b", ptx)),
        "barrier.sync": len(re.findall(r"\bbarrier\.sync\b", ptx)),
        "st.shared (any)": len(re.findall(r"\bst\.shared\.", ptx)),
        "ld.shared (any)": len(re.findall(r"\bld\.shared\.", ptx)),
    }
    passed = (counts["st.shared (any)"] > 0 and counts["ld.shared (any)"] > 0
              and (counts["bar.sync"] + counts["barrier.sync"]) > 0)
    return ptx, counts, passed


def reduce_scalar(ctx, value, op):
    out = np.zeros(1)
    ctx["cart_comm"].Allreduce(np.array([float(value)]), out, op=op)
    return float(out[0])


def run_solver(github_user: str):
    ctx = init_distributed_context(github_user)
    rank, ny, nx = ctx["rank"], ctx["ny"], ctx["nx"]

    log0(ctx, "=" * 78)
    log0(ctx, "Distributed MPI + CUDA anisotropic reaction-diffusion solver")
    log0(ctx, "=" * 78)
    log0(ctx, f"GitHub identity      : {ctx['github_user']}")
    log0(ctx, f"GPU                  : {ctx['gpu_name']}")
    log0(ctx, f"GPU UUID             : {ctx['gpu_uuid']}")
    log0(ctx, f"INSTANCE_SEED        : {ctx['instance_seed']} (0x{ctx['instance_seed']:016x})")
    log0(ctx, f"MPI library          : {MPI.Get_library_version().splitlines()[0].strip()}")
    log0(ctx, f"MPI ranks            : {ctx['size']} in a {ctx['dims'][0]}x{ctx['dims'][1]} "
              f"Cartesian grid, periods={ctx['periods']}")

    table = rank_table(ctx)
    log0(ctx, "\n[Task 1] Cartesian topology (PROC_NULL shown as '-'):")
    log0(ctx, "  rank  coords   up  down  left  right   local_shape   pid")
    for r in range(ctx["size"]):
        t = table[r].astype(int)
        f = lambda v: " -" if v < 0 else f"{v:2d}"
        log0(ctx, f"  {r:4d}  ({t[0]},{t[1]})   {f(t[2])}   {f(t[3])}    {f(t[4])}    {f(t[5])}"
                  f"    ({t[6]}, {t[7]})   {t[8]}")

    # ---- Task 2
    params = seeded_parameters(ctx["instance_seed"])
    fields = initialize_subdomains(ctx, params)
    kappa_max = reduce_scalar(ctx, fields["kappa"].max(), MPI.MAX)
    kappa_min = reduce_scalar(ctx, fields["kappa"].min(), MPI.MIN)
    owned = (slice(1, ny + 1), slice(1, nx + 1))
    src_integral = reduce_scalar(ctx, fields["src"][owned].sum() * fields["dx"] * fields["dy"], MPI.SUM)
    dx = fields["dx"]
    dt_max = dx * dx / (4.0 * kappa_max)
    dt = CFL_SAFETY * dt_max
    log0(ctx, f"\n[Task 2] Seeded subdomain initialisation (rank-local {ny + 2}x{nx + 2} arrays only)")
    log0(ctx, f"  dx = dy = 1/{GLOBAL_NX + 1} = {dx:.10e}")
    log0(ctx, f"  kappa range (global Allreduce MIN/MAX) = [{kappa_min:.6f}, {kappa_max:.6f}]")
    log0(ctx, f"  pulses: centres x={np.round(params['pulse_cx'], 4).tolist()} "
              f"y={np.round(params['pulse_cy'], 4).tolist()}")
    log0(ctx, f"          amplitudes={np.round(params['pulse_amp'], 4).tolist()} "
              f"sigmas={np.round(params['pulse_sigma'], 4).tolist()}")
    log0(ctx, f"  source dipole amp={params['src_amp']:.4f} src@{np.round(params['src_pos'], 4).tolist()} "
              f"sink@{np.round(params['sink_pos'], 4).tolist()}  integral S dA = {src_integral:.3e}")
    log0(ctx, f"  CFL: dt_max = dx^2/(4 kappa_max) = {dt_max:.6e}; dt = {CFL_SAFETY} * dt_max = {dt:.6e}")

    # ---- Task 3/4 resources
    bufs = allocate_exchange_buffers(ctx)
    streams = {"boundary": cuda.stream(), "interior": cuda.stream()}

    # Warm-up (JIT compilation) for both paths, not timed
    run_mode(ctx, fields, dt, bufs, streams, "serialized", WARMUP_STEPS, record=False)
    run_mode(ctx, fields, dt, bufs, streams, "overlapped", WARMUP_STEPS, record=False)

    log0(ctx, f"\n[Task 5] Running {STEPS} steps per mode (diagnostics every {DIAG_EVERY} steps)")
    results = {}
    for mode in ("serialized", "overlapped"):
        state, trace, wall_ms, timers = run_mode(ctx, fields, dt, bufs, streams, mode, STEPS)
        results[mode] = {"state": state, "trace": trace, "wall_ms": wall_ms, "timers": timers}
        log0(ctx, f"\n  -- {mode} mode --")
        log0(ctx, "     step        energy E (Allreduce)        L2 residual           max|u|")
        for d in trace:
            log0(ctx, f"     {d['step']:4d}   {d['energy']:.15e}   {d['l2_residual']:.12e}   {d['u_max']:.10f}")

    # Parity between serialized and overlapped final states (rank-local, then Allreduce MAX)
    u_ser = results["serialized"]["state"]["d_u"].copy_to_host()
    u_ovl = results["overlapped"]["state"]["d_u"].copy_to_host()
    parity_max_diff = reduce_scalar(ctx, np.abs(u_ser[owned] - u_ovl[owned]).max(), MPI.MAX)
    checksum = reduce_scalar(ctx, float((u_ovl[owned] ** 2).sum()), MPI.SUM)

    # Timing: slowest rank defines wall time; CUDA event times averaged and maxed over ranks
    serialized_ms = reduce_scalar(ctx, results["serialized"]["wall_ms"], MPI.MAX)
    overlapped_ms = reduce_scalar(ctx, results["overlapped"]["wall_ms"], MPI.MAX)
    ev_ser = results["serialized"]["timers"]
    ev_ovl = results["overlapped"]["timers"]
    cuda_events = {}
    for name, val in [("serialized_step_total_ms", ev_ser["full_ms"]),
                      ("overlapped_stageA_pack_d2h_ms", ev_ovl["pack_ms"]),
                      ("overlapped_interior_kernel_ms", ev_ovl["interior_ms"]),
                      ("overlapped_stageC_boundary_ms", ev_ovl["boundary_ms"])]:
        cuda_events[name] = {
            "mean_over_ranks": reduce_scalar(ctx, val, MPI.SUM) / ctx["size"],
            "max_over_ranks": reduce_scalar(ctx, val, MPI.MAX),
            "per_step_mean_ms": reduce_scalar(ctx, val, MPI.SUM) / ctx["size"] / STEPS,
        }

    log0(ctx, "\n[Task 5] Parity & benchmark")
    log0(ctx, f"  parity_max_diff (serialized vs overlapped) = {parity_max_diff:.3e}  "
              f"-> {'PASS' if parity_max_diff < 1e-12 else 'FAIL'} (< 1e-12)")
    log0(ctx, f"  global sum(u^2) checksum                   = {checksum:.15e}")
    log0(ctx, f"  serialized_ms = {serialized_ms:.3f} ms  ({serialized_ms / STEPS:.4f} ms/step)")
    log0(ctx, f"  overlapped_ms = {overlapped_ms:.3f} ms  ({overlapped_ms / STEPS:.4f} ms/step)")
    log0(ctx, f"  overlap speed-up = {serialized_ms / overlapped_ms:.4f}x")
    log0(ctx, f"  CUDA event timings (sum over {STEPS} steps, mean over ranks):")
    for k, v in cuda_events.items():
        log0(ctx, f"    {k:34s} {v['mean_over_ranks']:10.3f} ms  (max rank {v['max_over_ranks']:.3f} ms, "
                  f"{v['per_step_mean_ms'] * 1e3:.2f} us/step)")

    # PTX audit (rank 0 inspects the compiled kernel)
    if rank == 0:
        ptx, counts, ptx_passed = ptx_audit()
        with open("tiled_stencil_kernel.ptx", "w", encoding="utf-8") as fh:
            fh.write(ptx)
        log0(ctx, "\n[Task 5] PTX audit of tiled_stencil_kernel.inspect_asm()")
        for k, v in counts.items():
            log0(ctx, f"  {k:18s} : {v}")
        log0(ctx, f"  ptx_audit_passed   : {ptx_passed}")

        trace_ser = results["serialized"]["trace"]
        trace_ovl = results["overlapped"]["trace"]
        energies = [d["energy"] for d in trace_ovl]
        monotonic = all(b <= a for a, b in zip(energies, energies[1:]))
        with open(os.path.abspath(__file__), "rb") as fh:
            solver_sha = hashlib.sha256(fh.read()).hexdigest()
        import numba
        import mpi4py
        manifest = {
            "github_username": ctx["github_user"],
            "gpu_name": ctx["gpu_name"],
            "gpu_uuid": ctx["gpu_uuid"],
            "instance_seed": ctx["instance_seed"],
            "instance_seed_hex": f"0x{ctx['instance_seed']:016x}",
            "mpi_ranks": ctx["size"],
            "cart_dims": ctx["dims"],
            "cart_periods": ctx["periods"],
            "global_grid": [GLOBAL_NY, GLOBAL_NX],
            "local_padded_shape": [ny + 2, nx + 2],
            "tile_dim": TILE_DIM,
            "shared_dim": SHARED_DIM,
            "steps": STEPS,
            "lambda_damp": LAMBDA_DAMP,
            "dx": dx,
            "kappa_min": kappa_min,
            "kappa_max": kappa_max,
            "dt_max": dt_max,
            "dt": dt,
            "cfl_safety": CFL_SAFETY,
            "source_integral": src_integral,
            "seeded_parameters": params,
            "energy_trace_serialized": trace_ser,
            "energy_trace_overlapped": trace_ovl,
            "energy_monotonic_non_increasing": monotonic,
            "parity_max_diff": parity_max_diff,
            "parity_passed": parity_max_diff < 1e-12,
            "final_sum_u2_checksum": checksum,
            "serialized_ms": serialized_ms,
            "overlapped_ms": overlapped_ms,
            "overlap_speedup": serialized_ms / overlapped_ms,
            "cuda_event_timings_ms": cuda_events,
            "ptx_counts": counts,
            "ptx_audit_passed": ptx_passed,
            "rank_topology": [
                {"rank": r, "coords": [int(table[r, 0]), int(table[r, 1])],
                 "up": int(table[r, 2]), "down": int(table[r, 3]),
                 "left": int(table[r, 4]), "right": int(table[r, 5]),
                 "local_shape": [int(table[r, 6]), int(table[r, 7])]}
                for r in range(ctx["size"])
            ],
            "versions": {
                "python": sys.version.split()[0],
                "numpy": np.__version__,
                "numba": numba.__version__,
                "mpi4py": mpi4py.__version__,
                "mpi_library": MPI.Get_library_version().splitlines()[0].strip(),
            },
            "solver_sha256": solver_sha,
            "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        body = json.dumps(manifest, sort_keys=True).encode("utf-8")
        manifest["manifest_sha256"] = hashlib.sha256(body).hexdigest()
        with open("submission_manifest.json", "w", encoding="utf-8") as fh:
            json.dump(manifest, fh, indent=2)
        log0(ctx, f"\n  energy monotonic non-increasing (overlapped) : {monotonic}")
        log0(ctx, f"  submission_manifest.json written (sha256 {manifest['manifest_sha256'][:16]}...)")
        all_ok = ptx_passed and parity_max_diff < 1e-12 and ctx["size"] == 4
        log0(ctx, f"\nALL TASKS COMPLETE: {'YES' if all_ok else 'NO'}")
    ctx["cart_comm"].Barrier()


if __name__ == "__main__":
    if len(sys.argv) < 2 or not sys.argv[1].strip():
        raise SystemExit("usage: mpirun -np 4 python3 distributed_cuda_solver.py <github_username>")
    run_solver(sys.argv[1].strip())


Writing distributed_cuda_solver.py


In [ ]:
# CELL 4 - MULTI-RANK MPI + CUDA EXECUTION LAUNCHER
import shutil, subprocess, sys

if shutil.which("mpirun"):                      # Colab / OpenMPI (instructor command)
    launcher = ["mpirun", "--allow-run-as-root", "--oversubscribe", "-np", "4"]
    python = "python3"
else:                                           # Windows / MS-MPI local validation
    launcher = ["mpiexec", "-n", "4"]
    python = sys.executable
cmd = launcher + [python, "distributed_cuda_solver.py", GITHUB_USERNAME]

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
with open("solver_stdout.txt", "w", encoding="utf-8") as fh:
    fh.write(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

Distributed MPI + CUDA anisotropic reaction-diffusion solver
GitHub identity      : Stierlitzx
GPU                  : Tesla T4, 7.5
GPU UUID             : GPU-198f4157-3506-5952-ab80-80b6dd077a59
INSTANCE_SEED        : 15753744789197117896 (0xdaa08b6b3a4159c8)
MPI library          : Open MPI v4.1.6, package: Debian OpenMPI, ident: 4.1.6, repo rev: v4.1.6, Sep 30, 2023 
MPI ranks            : 4 in a 2x2 Cartesian grid, periods=[False, False]

[Task 1] Cartesian topology (PROC_NULL shown as '-'):
  rank  coords   up  down  left  right   local_shape   pid
     0  (0,0)    -    2     -     1    (258, 258)   969
     1  (0,1)    -    3     0     -    (258, 258)   970
     2  (1,0)    0    -     -     3    (258, 258)   971
     3  (1,1)    1    -     2     -    (258, 258)   972

[Task 2] Seeded subdomain initialisation (rank-local 258x258 arrays only)
  dx = dy = 1/513 = 1.9493177388e-03
  kappa range (global Allreduce MIN/MAX) = [0.080001, 0.239998]
  pulses: centres x=[0.3375, 0.6478, 0.50

In [ ]:
# Manifest verification summary (reads the file produced by rank 0)
import json
m = json.load(open("submission_manifest.json", encoding="utf-8"))
checks = {
    "GITHUB_USERNAME matches": m["github_username"] == GITHUB_USERNAME,
    "mpi_ranks == 4 in 2x2 grid": m["mpi_ranks"] == 4 and m["cart_dims"] == [2, 2],
    "ptx_audit_passed": m["ptx_audit_passed"],
    "parity_max_diff < 1e-12": m["parity_max_diff"] < 1e-12,
}
for k, v in checks.items():
    print(f"{k:30s} {'OK' if v else 'FAIL'}")
print()
for k in ["gpu_name", "gpu_uuid", "instance_seed", "kappa_max", "dt_max", "dt", "parity_max_diff",
          "serialized_ms", "overlapped_ms", "overlap_speedup", "ptx_counts",
          "energy_monotonic_non_increasing", "manifest_sha256"]:
    print(f"{k:32s} {m[k]}")
assert all(checks.values()), "manifest verification failed"
print("\n----- submission_manifest.json (verbatim) -----")
print(open("submission_manifest.json", encoding="utf-8").read())

GITHUB_USERNAME matches        OK
mpi_ranks == 4 in 2x2 grid     OK
ptx_audit_passed               OK
parity_max_diff < 1e-12        OK

gpu_name                         Tesla T4, 7.5
gpu_uuid                         GPU-198f4157-3506-5952-ab80-80b6dd077a59
instance_seed                    15753744789197117896
kappa_max                        0.23999840057536645
dt_max                           3.958192677177494e-06
dt                               3.562373409459745e-06
parity_max_diff                  0.0
serialized_ms                    3869.6389799999997
overlapped_ms                    1627.1544840000001
overlap_speedup                  2.3781632402151187
ptx_counts                       {'st.shared.f64': 5, 'ld.shared.f64': 5, 'bar.sync': 1, 'barrier.sync': 0, 'st.shared (any)': 5, 'ld.shared (any)': 5}
energy_monotonic_non_increasing  True
manifest_sha256                  77d3f7a8d6e122b2755a383a1a060c8afabec0adce932dd06618a79960c311af

----- submission_manifest.json (verbatim) -

In [ ]:
# Excerpt of the audited PTX: every shared-memory store/load and the barrier
import re
ptx = open("tiled_stencil_kernel.ptx", encoding="utf-8").read()
for line in ptx.splitlines():
    if re.search(r"\b(st|ld)\.shared\.|\bbar(rier)?\.sync\b", line):
        print(line.strip())

st.shared.f64 	[%rd48], %fd5;
st.shared.f64 	[%rd63], %fd6;
st.shared.f64 	[%rd78], %fd7;
st.shared.f64 	[%rd92], %fd8;
st.shared.f64 	[%rd106+136], %fd9;
bar.sync 	0;
ld.shared.f64 	%fd23, [%rd156+288];
ld.shared.f64 	%fd24, [%rd116];
ld.shared.f64 	%fd27, [%rd162];
ld.shared.f64 	%fd31, [%rd167+16];
ld.shared.f64 	%fd34, [%rd172];


### Analytical & Forensic Engineering Responses

All numbers below come from the Colab run shown above: **Tesla T4 (CC 7.5), UUID `GPU-198f4157-3506-5952-ab80-80b6dd077a59`,
`INSTANCE_SEED = 15753744789197117896 (0xdaa08b6b3a4159c8)`, Open MPI 4.1.6, 4 ranks in a 2 × 2 grid.**

---

- **Question 1 (Memory Coalescing & Strided Column Packing):**
  The padded array is `float64[258, 258]` in C order, so one row is 258 × 8 = **2,064 bytes** and element `u[i, j]` lives at
  byte `8·(258·i + j)`. The top/bottom strips `u[1, 1:257]` and `u[256, 1:257]` are 2,048 contiguous bytes, but the left/right
  strips `u[1:257, 1]` and `u[1:257, 256]` are 256 doubles spaced 2,064 bytes apart. A copy engine (`cudaMemcpyAsync`) moves one
  contiguous byte range, so it cannot fetch only those 256 values in one 1D DMA; it would have to copy the whole span between them.
  Inside `pack_halos_kernel`, a warp of 32 threads reading the row strip touches 32 consecutive doubles = 256 B = 8 32-byte sectors
  (100 % sector efficiency). The same warp reading a column touches 32 addresses 2,064 B apart, i.e. **32 separate sectors that each
  carry only 8 useful bytes (25 % efficiency)**. That uncoalesced read only happens once, on the GPU, in L2/DRAM. The writes into
  `send_buf[2..3, k]` are fully coalesced, and PCIe then carries one contiguous 8 KiB block (`h_send`, 4 × 256 × 8 B).
  **Cost of an unstrided bounding-box copy instead:** the smallest contiguous span holding column 1 runs from flat index 259
  (`u[1,1]`) to 66,049 (`u[256,1]`), which is **65,791 doubles = 526,328 B for 2,048 useful bytes, so 524,280 extra bytes per column**.
  Both columns add **1,048,560 B per rank per step** (≈ 87 µs at a sustained 12 GB/s Gen3 ×16, on top of the measured 209.56 µs/step
  for the whole Stage A). Across 4 ranks × 400 steps that is **1,677,696,000 extra bytes (≈ 1.68 GB) device→host**, and the same again
  host→device if the ghost columns were injected the same way. Our packed path moves **13,107,200 B** D2H in total for the whole run.

- **Question 2 (Shared-Memory Tile Indexing & Bank Conflict Analysis):**
  In the kernel `sy = ty + 1`, `sx = tx + 1` and `smem` is `float64[18, 18]`, so `smem[sy, sx]` sits at byte `8·(18·sy + sx)`.
  For a full interior tile:

  | thread | writes (before `bar.sync`) | reads (after `bar.sync`) |
  |---|---|---|
  | `(tx=0, ty=0)` | `smem[1,1]` ← `u[i,j]`, `smem[0,1]` ← `u[i-1,j]` (top apron), `smem[1,0]` ← `u[i,j-1]` (left apron) | `smem[1,1]`, `smem[0,1]`, `smem[2,1]`, `smem[1,0]`, `smem[1,2]` |
  | `(tx=15, ty=15)` | `smem[16,16]` ← `u[i,j]`, `smem[17,16]` ← `u[i+1,j]` (bottom apron), `smem[16,17]` ← `u[i,j+1]` (right apron) | `smem[16,16]`, `smem[15,16]`, `smem[17,16]`, `smem[16,15]`, `smem[16,17]` |

  The four corners `smem[0,0]`, `smem[0,17]`, `smem[17,0]`, `smem[17,17]` are never touched (5-point stencil).
  Bank mapping on CC 7.5: 32 banks × 4 B, and a `float64` covers two adjacent banks, `2·(18·sy+sx) mod 32` and `+1`.
  A 16 × 16 block puts rows `ty = 2w, 2w+1` in warp `w`. A 64-bit warp access (256 B) is always served as two half-warp passes of
  128 B, which is the shared-memory width per cycle and not a conflict. **Row-wise accesses** (`smem[sy, sx]`, `smem[sy±1, sx]`, and
  the shifted `smem[sy, sx±1]`) give each half-warp 16 consecutive doubles = 128 contiguous bytes that hit all 32 banks exactly once,
  so they are **conflict-free**. **Column-wise accesses** would conflict: consecutive rows are 18 doubles = 36 words apart, so the
  bank of row k is `4k mod 32`. Rows k and k+8 then land on the same bank pair, and a half-warp reading 16 rows of one column has a
  **2-way bank conflict**. Our kernel never does this; its only column-direction accesses are the `tx==0` / `tx==15` apron stores,
  2 threads per warp at bank offset 4, so no conflict. An odd tile width (17 or 19 doubles, row pitch 34 or 38 words) would make
  column access conflict-free as well.

- **Question 3 (PTX Assembly Audit & Warp Synchronization):**
  `tiled_stencil_kernel.inspect_asm()` in this run contains **`st.shared.f64` = 5, `ld.shared.f64` = 5, `bar.sync` = 1**
  (`barrier.sync` = 0), so `ptx_audit_passed = true`. The counts map one-to-one onto the source: 5 guarded stores (centre + 4 apron
  edges), one `bar.sync 0` from `cuda.syncthreads()`, and 5 loads (centre + N/S/W/E). The offsets confirm the layout:
  `st.shared.f64 [%rd106+136]` is +17 doubles (right apron column), `ld.shared.f64 [%rd156+288]` is +36 doubles = two 18-wide rows
  (south neighbour relative to north), and `[%rd167+16]` is +2 doubles (east relative to west).
  **Hazard without the barrier:** this is a cross-warp read-after-write race. Warp `w` owns tile rows `ty = 2w, 2w+1`, but its stencil reads
  `smem[sy-1, sx]` and `smem[sy+1, sx]`, which belong to the rows of warps `w-1` and `w+1`. Apron row 0 is written only by warp 0
  (`ty==0`) and apron row 17 only by warp 7 (`ty==15`). The 8 warps are scheduled independently, so without `bar.sync` warp 3 can issue
  its `ld.shared` before warps 2 and 4 have executed their `st.shared`. It then reads stale data left in that shared-memory region by a
  previous block. The update is silently wrong and changes between runs, so the parity check (here exactly **0.0**) would fail. One
  barrier is enough because the tile is written once and only read afterwards (no write-after-read reuse).

- **Question 4 (Multi-Process Context Sharing on a Single Physical GPU):**
  `mpirun -np 4` creates 4 OS processes (PIDs 969–972 in the topology table), each with its own CUDA context on the one T4.
  Without MPS, contexts from different processes do not run kernels concurrently. The GPU time-slices between them, and each
  switch saves/restores context state. Inside one context, our two streams can overlap, and the T4's copy engines can run DMA
  in parallel with kernels of the active context. All 4 ranks share those 40 SMs, the 2 copy engines and one PCIe link.
  **Measured:** `serialized_ms = 3869.639 ms` (9.674 ms/step) vs `overlapped_ms = 1627.154 ms` (4.068 ms/step), a **2.378×
  speed-up**, with parity diff **0.0**. The CUDA events show that compute is tiny: the 254 × 254 interior kernel costs **298.33 µs/step**
  and Stage A (pack + D2H + sync) **209.56 µs/step**. The serialized step spans **9,338 µs** between its events, so over 90 % of a
  serialized step is waiting: host synchronisation, MPI hand-shakes with the slowest neighbour, and context time-slices. Stage C
  in overlapped mode (Waitall + H2D + unpack + 4 ring launches) still accounts for **3,191 µs/step**, and that is mostly the MPI wait. The
  overlapped schedule wins mainly because each rank enqueues its interior kernel before blocking in `Waitall`. The GPU then works
  through the ranks' interior kernels while the processes wait for each other, and only the thin boundary ring stays on the
  critical path after communication. On one shared GPU the gain therefore comes from hiding latency and scheduling gaps, not
  from extra hardware, and it is capped by the fact that all four ranks' kernels still go through the same SMs one context at a time.
  **With four dedicated GPUs over InfiniBand,** each rank has its own SMs and copy engines and no time-slicing. Ideal overlap then
  hides the network term behind the interior compute, so the step time approaches `max(T_interior, T_comm) + T_ring`. The gain is bounded by
  `T_comm / (T_interior + T_comm)` and grows with subdomain size. With GPUDirect RDMA the pinned host staging disappears completely.

- **Question 5 (Point-to-Point Message Deadlock & `MPI.PROC_NULL` Semantics):**
  A blocking `Send` returns as soon as the user buffer can be reused. **Below the transport's eager limit** (a few KiB for Open MPI's
  shared-memory BTL; our strips are 256 × 8 = **2,048 B**), the library copies the message into an internal buffer and `Send` returns
  at once, so the naive code happens to work on small grids. **Above the eager limit** it switches to the **rendezvous** protocol:
  the sender sends a request-to-send and blocks until the receiver posts the matching `Recv` and replies clear-to-send. If every rank
  executes the same fixed order, e.g. `Send(up); Send(down); Send(left); Send(right); Recv(...)`, then rank 0 `(0,0)` blocks in
  `Send(down → 2)` waiting for rank 2 to post `Recv(up)`. Rank 2 is blocked in its own `Send(up → 0)` waiting for rank 0 to post
  `Recv(down)`. Neither reaches its `Recv`, and the same holds for the 0↔1 and 2↔3 pairs. That is a circular wait: a deadlock that
  appears only when the grid is large enough (e.g. > 512 columns at a 4 KiB limit) to push the halos over the threshold. Our solver
  avoids this by posting all 4 `Irecv` before the 4 `Isend` and completing them with one `MPI.Request.Waitall`, so no call blocks before
  all requests are posted. (`Sendrecv` or odd/even ordering would also work.)
  **`MPI.PROC_NULL`:** `Create_cart(periods=[False, False])` makes `Shift` return `PROC_NULL` beyond the global edge. Rank 0 at
  `(0,0)` gets `up = left = PROC_NULL` and `down = 2`, `right = 1`, exactly as printed in the Task 1 table. A send to or receive from
  `PROC_NULL` completes immediately, transfers nothing and leaves the buffer untouched. So every rank, corners included, runs the
  **same 8 calls with no `if` on its position**, and two of rank 0's eight requests are free no-ops. The unpack kernel receives
  `has_up/has_left = 0` for those edges, so the ghost row/column keeps its initial **u = 0**, which is the homogeneous Dirichlet condition.

- **Question 6 (Explicit CFL Stability & Energy Conservation Proof):**
  From the run: `kappa_max = 0.2399984006` (Allreduce MAX; range `[0.080001, 0.239998]`), `Δx = Δy = 1/513 = 1.9493177388e-3`,
  `Δx² = 3.7998396e-6`.
  `Δt_max = Δx² / (4·κ_max) = 3.7998396e-6 / 0.9599936 = **3.958193e-6**`. The run used `Δt = 0.9·Δt_max = **3.562373e-6**`, so the
  diffusion number is `κ_max·Δt/Δx² = 0.225 < 0.25`. The interface conductivities are arithmetic means of κ, so none exceeds κ_max
  and the bound is conservative. The reaction term adds `3λu²·Δt ≤ 3·0.15·1.3204²·3.56e-6 = 2.8e-6`, which is negligible.
  400 steps simulate `t = 1.424949e-3`.

  | step | E = Σ u Δx Δy (Allreduce) | ΔE over previous 100 steps | L2 residual ‖uⁿ−uⁿ⁻¹‖₂ | max\|u\| |
  |---:|---:|---:|---:|---:|
  | 0 | 7.606762073269764e-02 | — | — | 1.3203510596 |
  | 100 | 7.606569852616432e-02 | −1.922207e-06 | 2.426829e-02 | 1.2822818371 |
  | 200 | 7.606388322666909e-02 | −1.815299e-06 | 2.284185e-02 | 1.2462127205 |
  | 300 | 7.606216093689147e-02 | −1.722290e-06 | 2.159952e-02 | 1.2121959096 |
  | 400 | 7.606052255966575e-02 | −1.638377e-06 | 2.050234e-02 | 1.1798519187 |

  E decreases strictly at every checkpoint (total −7.098e-6, relative −9.33e-5), and the step residual and max|u| also fall
  monotonically. These are signs of a stable, dissipative scheme; an unstable explicit step would make the residual grow
  geometrically. **Energy budget:** summing the update over all cells, the flux-form diffusion terms cancel pairwise between
  neighbours (the scheme is conservative), so `dE/dt = (boundary flux) − λ Σ u³ ΔxΔy + Σ S ΔxΔy`. The seeded source is a
  zero-net dipole (printed `∫S dA = 0.000e+00`). The pulses sit ≥ 6σ from the walls, so the boundary flux is ~1e-8. That leaves
  `dE/dt ≈ −λ∫u³ < 0`, which explains the monotone decay. For the three seeded Gaussians, `∫u³ = Σ A³·2πσ²/3 = 0.036602`, which
  predicts `ΔE ≈ −0.15 · 0.036602 · (100·Δt) = −1.956e-6` over steps 0→100. That agrees with the measured **−1.922e-6** to 1.7 %.
  The gradual drop in |ΔE| (1.92 → 1.64 e-6) follows the cubic damping as the peak amplitude decays from 1.32 to 1.18.
  Serialized and overlapped traces are identical to all 16 printed digits.
